Purpose: Create supplemental table containing important information for all pathways' genes.<br>
Author: Anna Pardo<br>
Date initiated: July 16, 2026

In [1]:
import pandas as pd
import numpy as np
import os
import json

In [2]:
# load pathway gene information
prncit = pd.read_csv("./photresp_N_citrate_genes_Yucca.csv",sep=",",header="infer")
clock = pd.read_csv("./circadian_light_genes_by_orthology_Yucca.csv",sep=",",header="infer")
cam = pd.read_csv("./camgenes_Ya_Yf_orthology_synteny.txt",sep="\t",header="infer")

In [3]:
# prncit contains CAM genes!!! drop those out to avoid duplicated gene names
prncit = prncit[~prncit["Pathway"].isin(["CAM-dark","CAM-light"])]

In [4]:
cam.head()

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome
0,Yucal.01G165600.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_1,Ya
1,Yucal.02G112700.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_2,Ya
2,Yucal.04G001000.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_1,Ya
3,Yucal.07G000800.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_2,Ya
4,Yucal.03G120900.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_1,Ya


In [5]:
prncit.head()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
0,PhotResp,PGP,Yucal.01G302600.v2.1,Ya,Ya_PGP_1
1,PhotResp,PGP,Yucal.22G052300.v2.1,Ya,Ya_PGP_2
2,PhotResp,PGP,YufilH1006999m.g,Yf,Yf_PGP_1
3,PhotResp,PGP,YufilH1040717m.g,Yf,Yf_PGP_2
4,PhotResp,PGP,YufilH1084368m.g,Yf,Yf_PGP_3


In [6]:
clock.head()

,GeneID,Orthogroup,gene_name,subgenome,gene_name_unique
0,Yucal.11G006100.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_1
1,Yucal.15G098200.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_2
2,Yucal.16G106200.v2.1,OG0002937,TOC1,Ya,Ya_TOC1_3
3,YufilH1057026m.g,OG0002937,TOC1,Yf,Yf_TOC1_1
4,YufilH1057027m.g,OG0002937,TOC1,Yf,Yf_TOC1_2


In [7]:
cam = cam[["GeneID","gene_abbr","subgenome","gene_abbr_unique","Pathway"]].rename(columns={"gene_abbr":"Gene Family",
                                                                                "subgenome":"Parental Origin",
                                                                                "gene_abbr_unique":"Gene Name"})
cam.head()

,GeneID,Gene Family,Parental Origin,Gene Name,Pathway
0,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,CAM-dark
1,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,CAM-dark
2,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,CAM-dark
3,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,CAM-dark
4,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,CAM-dark


In [8]:
clock = clock[["GeneID","gene_name","subgenome","gene_name_unique"]].rename(columns={"gene_name":"Gene Family",
                                                                                    "subgenome":"Parental Origin",
                                                                                    "gene_name_unique":"Gene Name"})

In [9]:
prncit = prncit[["GeneID","gene_family","subgenome","gene_name_unique","Pathway"]].rename(columns={"gene_family":"Gene Family",
                                                                                                  "subgenome":"Parental Origin",
                                                                                                  "gene_name_unique":"Gene Name"})

In [10]:
clock["Pathway"] = "Clock"

In [11]:
# stick all three dataframes together
pathinfo = pd.concat([cam, clock, prncit], ignore_index=True)

In [12]:
pathinfo = pathinfo[["Pathway","GeneID","Gene Family","Parental Origin","Gene Name"]]

In [13]:
pathinfo["Pathway"].unique()

array(['CAM-dark', 'CAM-light', 'Clock', 'PhotResp', 'N-metab',
       'CA-cycle'], dtype=object)

In [14]:
# load remaining information: TSGs, DEGs, HybridExpress, ASE results, polynomial modeling; incorporate into dataframe
## start with TS gene information
c3camts = pd.read_csv("/home/leviathan22/yucca-genomics/masigpro/C3+CAM_TSgenes_9gt_e05.txt",sep="\t",header="infer")
camts = pd.read_csv("/home/leviathan22/yucca-genomics/masigpro/CAM_TSgenes_9gt_e05.txt",sep="\t",header="infer")
faccamts = pd.read_csv("/home/leviathan22/yucca-genomics/masigpro/facCAM_TSgenes_9gt_e05.txt",sep="\t",header="infer")

In [15]:
yats = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/masigpro_results/Ya_hclust_k6_clusters_01-Jul-2026.txt",
                  sep="\t",header="infer")
yfts = pd.read_csv("/home/leviathan22/yucca-genomics/masigpro/Yf_3reps_masigpro_clusters_Jul02.txt",
                  sep="\t",header="infer")

In [16]:
# assemble a quick dict of TSGs
tsg_lists = {"Ya":list(yats["GeneID"]),
            "Yf":list(yfts["GeneID"]),
            "CAM":list(camts["GeneID"]),
            "C3+CAM":list(c3camts["GeneID"]),
            "facCAM":list(faccamts["GeneID"])}

In [17]:
# add TS information to the pathway gene information
tsdict = {"GeneID":[],"TS in Ya":[],"TS in Yf":[],"TS in CAM":[],"TS in C3+CAM":[],"TS in facCAM":[]}
for i in pathinfo["GeneID"].unique():
    tsdict["GeneID"].append(i)
    for k,v in tsg_lists.items():
        if i in v:
            tsdict["TS in "+k].append("Y")
        else:
            tsdict["TS in "+k].append("N")

pathinfo = pathinfo.merge(pd.DataFrame(tsdict),how="left")
pathinfo.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N


In [18]:
# load HybridExpress results
hybexpres = pd.read_csv("/home/leviathan22/yucca-genomics/differential_expression/expression_partitioning_downstream/HybridExpress_results_withphys_top30.txt",
                       sep="\t",header="infer")
hybexpres.head()

/tmp/ipykernel_7319/108307783.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  hybexpres = pd.read_csv("/home/leviathan22/yucca-genomics/differential_expression/expression_partitioning_downstream/HybridExpress_results_withphys_top30.txt",


,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat,phys
0,recip_syn1000,1,ADD,5.957692,-0.447832,18_D,18,D,C3+CAM
1,recip_syn10008,1,ADD,0.877055,-1.115803,18_D,18,D,C3+CAM
2,recip_syn10040,1,ADD,3.715160,-1.074124,18_D,18,D,C3+CAM
3,recip_syn10057,1,ADD,1.211627,-0.911991,18_D,18,D,C3+CAM
4,recip_syn10061,1,ADD,3.105884,-1.503516,18_D,18,D,C3+CAM


In [19]:
# load synteny data
synlong = pd.read_csv("/home/leviathan22/Yucca_genomics/yucca_synteny/reciprocal_syntelogs_sameinYaYf_long.txt",sep="\t",
                     header="infer")
synlong.head()

,GeneID,syntelogID
0,YufilH1000002m.g,recip_syn1
1,Yucal.01G000100.v2.1,recip_syn1
2,YufilH1000007m.g,recip_syn2
3,Yucal.01G000200.v2.1,recip_syn2
4,YufilH1000011m.g,recip_syn3


In [20]:
hybexpres = synlong.merge(hybexpres.rename(columns={"Gene":"syntelogID"}))
hybexpres.head()

,GeneID,syntelogID,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat,phys
0,YufilH1000007m.g,recip_syn2,7,DOWN,-0.430199,-0.214936,2AB_D,2AB,D,C3+CAM
1,YufilH1000007m.g,recip_syn2,8,UP,0.524021,0.744500,19_D,19,D,C3+CAM
2,YufilH1000007m.g,recip_syn2,4,ELD_P1,-0.024002,0.201819,43_D,43,D,CAM
3,YufilH1000007m.g,recip_syn2,4,ELD_P1,0.136697,0.361422,51_D,51,D,CAM
4,YufilH1000007m.g,recip_syn2,8,UP,0.518026,0.437338,19_W,19,W,C3+CAM


In [21]:
hybexpres["phys_treat"] = hybexpres["phys"]+"_"+hybexpres["treat"]

In [22]:
# make summary to append to the pathinfo dataframe
hedict = {"GeneID":[]}
for i in hybexpres["phys_treat"].unique():
    hedict[i] = []

In [23]:
for i in pathinfo["GeneID"].unique():
    if i in list(hybexpres["GeneID"].unique()):
        hedict["GeneID"].append(i)
        df = hybexpres[hybexpres["GeneID"]==i]
        pts = list(df["phys_treat"].unique())
        for j in hybexpres["phys_treat"].unique():
            if j in pts:
                dfsub = df[df["phys_treat"]==j]
                classes = list(dfsub["Class"].unique())
                cstr = ""
                for x in classes:
                    cstr = cstr+x+", "
                hedict[j].append(cstr.rstrip(", "))
            else:
                hedict[j].append("N")

In [24]:
pathinfo = pathinfo.merge(pd.DataFrame(hedict),how="left")
pathinfo.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM,C3+CAM_D,CAM_D,C3+CAM_W,CAM_W,facultative CAM_W,facultative CAM_D
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N,"UP, ELD_P1",UP,"ADD, ELD_P1, UP","UP, ELD_P2","ELD_P1, UP",UP
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N,ELD_P2,ELD_P2,"ADD, ELD_P2","ELD_P2, ADD, UP",ELD_P2,ELD_P2
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N,"DOWN, ELD_P2, ELD_P1","ELD_P2, DOWN","DOWN, ELD_P2, UP","ELD_P1, ELD_P2","ELD_P2, DOWN, ELD_P1","DOWN, ELD_P2, ADD"
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N,"ELD_P2, ELD_P1, UP, ADD","ELD_P2, ADD","DOWN, UP",DOWN,DOWN,"ELD_P1, ADD"
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N,DOWN,DOWN,"DOWN, ELD_P2","DOWN, ELD_P2",DOWN,DOWN


In [25]:
# add polynomial modeling results
## first, load polynomial modeling results
polymodres = pd.read_csv("./polymod3_res_allpath_3wayTSGs.txt",sep="\t",header="infer")
polymodres.head()

,GeneID,gene_family,GeneName,Model,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F),FDR_p,Description
0,Yucal.01G000900.v2.1,NaN,NaN,polyZT*treat*phys,803,477.122937,16.0,46.590076,4.900706,1.082390e-09,4.544456e-09,mTERF
1,Yucal.01G000900.v2.1,NaN,NaN,polyZT*phys,819,523.713013,-4.0,-37.009588,15.571825,2.812654e-12,1.863688e-11,mTERF
2,Yucal.01G000900.v2.1,NaN,NaN,polyZT*treat,815,486.703424,8.0,38.297529,8.056864,1.700071e-10,8.296607e-10,mTERF
3,Yucal.01G000900.v2.1,NaN,NaN,polyZT,823,525.000954,NaN,NaN,NaN,NaN,NaN,mTERF
4,Yucal.01G002900.v2.1,NaN,NaN,polyZT*phys,819,235.559548,-4.0,31.742931,NaN,NaN,NaN,Zinc finger family protein


In [26]:
# drop NAs from polymodres
pmr_nona = polymodres.dropna(subset="FDR_p")
pmr_nona.head()

,GeneID,gene_family,GeneName,Model,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F),FDR_p,Description
0,Yucal.01G000900.v2.1,NaN,NaN,polyZT*treat*phys,803,477.122937,16.0,46.590076,4.900706,1.082390e-09,4.544456e-09,mTERF
1,Yucal.01G000900.v2.1,NaN,NaN,polyZT*phys,819,523.713013,-4.0,-37.009588,15.571825,2.812654e-12,1.863688e-11,mTERF
2,Yucal.01G000900.v2.1,NaN,NaN,polyZT*treat,815,486.703424,8.0,38.297529,8.056864,1.700071e-10,8.296607e-10,mTERF
5,Yucal.01G002900.v2.1,NaN,NaN,polyZT*treat,815,267.302479,8.0,2.485856,1.081596,3.736967e-01,3.944805e-01,Zinc finger family protein
6,Yucal.01G002900.v2.1,NaN,NaN,polyZT*treat*phys,803,230.694182,16.0,4.865366,1.058460,3.917820e-01,4.128505e-01,Zinc finger family protein


In [27]:
pmod = {"GeneID":[]}
for i in polymodres["Model"].unique():
    if "*" in i:
        pmod[i] = []

In [28]:
pmod

{'GeneID': [], 'polyZT*treat*phys': [], 'polyZT*phys': [], 'polyZT*treat': []}

In [29]:
for i in pathinfo["GeneID"].unique():
    df = pmr_nona[pmr_nona["GeneID"]==i]
    pmod["GeneID"].append(i)
    for k,v in pmod.items():
        if k in df["Model"].unique():
            pval = df.loc[df['Model'] == k, 'FDR_p'].iloc[0]
            if pval < 0.05:
                v.append("sig")
            else:
                v.append("not sig")
        elif k!="GeneID":
            v.append("no result")

In [30]:
pathinfo = pathinfo.merge(pd.DataFrame(pmod),how="left")

In [31]:
pathinfo.head()

,Pathway,GeneID,Gene Family,Parental Origin,Gene Name,TS in Ya,TS in Yf,TS in CAM,TS in C3+CAM,TS in facCAM,C3+CAM_D,CAM_D,C3+CAM_W,CAM_W,facultative CAM_W,facultative CAM_D,polyZT*treat*phys,polyZT*phys,polyZT*treat
0,CAM-dark,Yucal.01G165600.v2.1,bCA1234,Ya,Ya_bCA1234_1,Y,N,N,N,N,"UP, ELD_P1",UP,"ADD, ELD_P1, UP","UP, ELD_P2","ELD_P1, UP",UP,sig,no result,sig
1,CAM-dark,Yucal.02G112700.v2.1,bCA1234,Ya,Ya_bCA1234_2,Y,N,N,N,N,ELD_P2,ELD_P2,"ADD, ELD_P2","ELD_P2, ADD, UP",ELD_P2,ELD_P2,not sig,no result,not sig
2,CAM-dark,Yucal.04G001000.v2.1,bCA5,Ya,Ya_bCA5_1,N,N,Y,N,N,"DOWN, ELD_P2, ELD_P1","ELD_P2, DOWN","DOWN, ELD_P2, UP","ELD_P1, ELD_P2","ELD_P2, DOWN, ELD_P1","DOWN, ELD_P2, ADD",sig,sig,sig
3,CAM-dark,Yucal.07G000800.v2.1,bCA5,Ya,Ya_bCA5_2,Y,N,N,N,N,"ELD_P2, ELD_P1, UP, ADD","ELD_P2, ADD","DOWN, UP",DOWN,DOWN,"ELD_P1, ADD",sig,sig,sig
4,CAM-dark,Yucal.03G120900.v2.1,NAD-MDH-cp,Ya,Ya_NAD-MDH-cp_1,Y,N,N,N,N,DOWN,DOWN,"DOWN, ELD_P2","DOWN, ELD_P2",DOWN,DOWN,not sig,no result,not sig


In [32]:
# load ASE information
## remember Ya is reference, so LFC>0 = Yf bias, LFC<0 = Ya bias
ase = pd.read_csv("./differential_expression/ASE_gttreat_output.txt",sep="\t",header="infer")
ase.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,GeneID,Contrast
0,67.211469,0.764897,0.089259,8.569367,1.040554e-17,4.892090e-17,recip_syn2,18_W_aloifolia-V-18_W_filamentosa
1,207.226297,-0.485237,0.071331,-6.802606,1.027436e-11,3.495057e-11,recip_syn3,18_W_aloifolia-V-18_W_filamentosa
2,43.434762,0.344017,0.106880,3.218711,1.287680e-03,2.321305e-03,recip_syn4,18_W_aloifolia-V-18_W_filamentosa
3,52.217998,-1.184873,0.103213,-11.479849,1.665721e-30,1.346501e-29,recip_syn5,18_W_aloifolia-V-18_W_filamentosa
4,390.928299,-0.320654,0.073987,-4.333917,1.464794e-05,3.195546e-05,recip_syn7,18_W_aloifolia-V-18_W_filamentosa


In [33]:
ase = synlong.merge(ase[["GeneID","Contrast","log2FoldChange"]].rename(columns={"GeneID":"syntelogID"}))
ase.head()

,GeneID,syntelogID,Contrast,log2FoldChange
0,YufilH1000007m.g,recip_syn2,18_W_aloifolia-V-18_W_filamentosa,0.764897
1,YufilH1000007m.g,recip_syn2,2AB_W_aloifolia-V-2AB_W_filamentosa,1.041007
2,YufilH1000007m.g,recip_syn2,2AB_D_aloifolia-V-2AB_D_filamentosa,0.825842
3,YufilH1000007m.g,recip_syn2,1AB_W_aloifolia-V-1AB_W_filamentosa,0.959057
4,YufilH1000007m.g,recip_syn2,18_D_aloifolia-V-18_D_filamentosa,0.924623


In [34]:
# load physiotype information
phys = json.load(open("./physiology/physiological_categories_from_TA.json"))

In [35]:
gt = []
treat = []
for i in list(ase["Contrast"]):
    gt.append(i.split("_")[0])
    treat.append(i.split("_")[1])
ase["genotype"] = gt
ase["treat"] = treat
ase.head()

,GeneID,syntelogID,Contrast,log2FoldChange,genotype,treat
0,YufilH1000007m.g,recip_syn2,18_W_aloifolia-V-18_W_filamentosa,0.764897,18,W
1,YufilH1000007m.g,recip_syn2,2AB_W_aloifolia-V-2AB_W_filamentosa,1.041007,2AB,W
2,YufilH1000007m.g,recip_syn2,2AB_D_aloifolia-V-2AB_D_filamentosa,0.825842,2AB,D
3,YufilH1000007m.g,recip_syn2,1AB_W_aloifolia-V-1AB_W_filamentosa,0.959057,1AB,W
4,YufilH1000007m.g,recip_syn2,18_D_aloifolia-V-18_D_filamentosa,0.924623,18,D


In [36]:
ase["phys"] = ase["genotype"].map(phys)

In [37]:
ase["phys_treat"] = ase["phys"]+"_"+ase["treat"]

In [38]:
bias = []
for i in list(ase["log2FoldChange"]):
    if i<0:
        bias.append("Ya")
    elif i>0:
        bias.append("Yf")
        
ase["bias"] = bias

In [39]:
ase_gtsum = ase.groupby(["GeneID","phys_treat","bias"]).count().reset_index()[["GeneID","phys_treat","bias","genotype"]].rename(columns={"genotype":"n_gts"})

In [40]:
# find number of total genotypes in each physiotype
ase[["phys","genotype"]].drop_duplicates().groupby("phys").count()

,genotype
phys,
C3+CAM,10
CAM,6
facultative CAM,5


In [41]:
nphys = {"CAM":6,"facultative CAM":5,"C3+CAM":10}

In [42]:
asedict = {"GeneID":[]}
for i in ase["phys_treat"].unique():
    asedict["ASE_"+i] = []

In [43]:
ase_gtsum.head()

,GeneID,phys_treat,bias,n_gts
0,Yucal.01G000200.v2.1,C3+CAM_D,Yf,9
1,Yucal.01G000200.v2.1,C3+CAM_W,Yf,10
2,Yucal.01G000200.v2.1,CAM_D,Yf,5
3,Yucal.01G000200.v2.1,CAM_W,Yf,6
4,Yucal.01G000200.v2.1,facultative CAM_D,Yf,3


In [44]:
for i in pathinfo["GeneID"].unique():
    if i in list(ase_gtsum["GeneID"].unique()):
        df = ase_gtsum[ase_gtsum["GeneID"]==i]
        asedict["GeneID"].append(i)
        for j in ase_gtsum["phys_treat"].unique():
            if j in list(df["phys_treat"].unique()):
                dfsub = df[df["phys_treat"]==j].sort_values(by="n_gts",ascending=False)
                b = dfsub.iloc[0,2]
                pct = round((dfsub.iloc[0,3]/nphys[j.split("_")[0]])*100,1)
                if len(dfsub.index)>1:
                    b2 = dfsub.iloc[1,2]+", "
                    lowpct = str(round((dfsub.iloc[1,3]/nphys[j.split("_")[0]])*100,1))+"%"
                    conj = "; "
                else:
                    b2 = ""
                    lowpct = ""
                    conj = ""
                asedict["ASE_"+j].append(b+", "+str(pct)+"%"+conj+b2+lowpct)
            else:
                asedict["ASE_"+j].append("no result")

In [45]:
pathinfo = pathinfo.merge(pd.DataFrame(asedict),how="left")

In [46]:
pathinfo.to_csv("./pathway_genes_results_summary.csv",sep=",",header=True,index=False)